# Real-Time YOLO Object Detection Inference (JetRacer Camera)

This notebook loads a trained **YOLO model** (e.g. `yolov8n.pt`, custom traffic sign model, or TensorRT `.engine`) and runs real-time object detection on the **JetRacer CSI Camera** feed.

### 1. Setup Environment & Load YOLO Model

In [ ]:
import os
import cv2
import time
import torch
import numpy as np
import ipywidgets
from IPython.display import display
from ipywidgets import Layout
from ultralytics import YOLO

model_path = 'yolov8n.pt'  # Change to your trained weights e.g. 'best.pt' or 'best.engine'

model = None
if os.path.exists(model_path):
    try:
        model = YOLO(model_path)
        print(f"Successfully loaded YOLO model weights from '{model_path}'")
    except Exception as e:
        print(f"Error loading YOLO model: {e}")
else:
    print(f"Notice: '{model_path}' not found locally. Loading standard YOLOv8n pretrained weights...")
    try:
        model = YOLO('yolov8n.pt')
        print("Loaded pretrained 'yolov8n.pt' model.")
    except Exception as e:
        print(f"Could not load YOLO model: {e}")


### 2. Initialize CSI Camera

In [ ]:
from jetcam.csi_camera import CSICamera
from jetcam.utils import bgr8_to_jpeg

camera = CSICamera(width=224, height=224, capture_fps=65)
camera.running = True
print(f"CSI Camera initialized successfully: {camera.width}x{camera.height}")


### 3. Real-Time YOLO Inference & Live Preview UI

* Adjust **Confidence Threshold** slider.
* Toggle **Inference Mode** to start/stop real-time object detection stream.
* Live stream overlays detected bounding boxes, class names, confidence scores, and real-time FPS.

In [ ]:
import threading
import cv2
import time
import ipywidgets
from IPython.display import display
from ipywidgets import Layout
from jetcam.utils import bgr8_to_jpeg

style_dict = {'description_width': '140px'}

# UI Controls
conf_slider = ipywidgets.FloatSlider(description='Confidence Threshold', min=0.1, max=0.9, value=0.25, step=0.05, readout=True, readout_format='.2f', layout=Layout(width='400px'), style=style_dict)
fps_display = ipywidgets.FloatText(description='FPS:', value=0.0, disabled=True, layout=Layout(width='200px'), style=style_dict)
latency_display = ipywidgets.FloatText(description='Latency (ms):', value=0.0, disabled=True, layout=Layout(width='220px'), style=style_dict)
detections_display = ipywidgets.IntText(description='Detections Count:', value=0, disabled=True, layout=Layout(width='220px'), style=style_dict)

state_widget = ipywidgets.ToggleButtons(options=['Off', 'Live YOLO Inference'], description='State:', value='Off')
yolo_preview_widget = ipywidgets.Image(value=bgr8_to_jpeg(camera.value), format='jpeg', width=camera.width, height=camera.height)
status_label = ipywidgets.Label(value='Ready for inference.')

yolo_live_active = False

def yolo_inference_loop():
    global yolo_live_active
    while yolo_live_active:
        try:
            start_t = time.time()
            frame = camera.value.copy()  # BGR frame
            
            if model is not None:
                # Run YOLO model inference
                results = model(frame, conf=conf_slider.value, verbose=False)
                
                # Render detection bounding boxes on frame
                annotated_frame = results[0].plot()
                det_count = len(results[0].boxes)
                detections_display.value = det_count
            else:
                annotated_frame = frame
                det_count = 0
            
            elapsed_ms = (time.time() - start_t) * 1000.0
            fps = 1000.0 / max(1.0, elapsed_ms)
            
            fps_display.value = round(fps, 1)
            latency_display.value = round(elapsed_ms, 1)
            
            # Update preview widget
            yolo_preview_widget.value = bgr8_to_jpeg(annotated_frame)
            
        except Exception as e:
            print(f"Error in YOLO inference loop: {e}")
            break
            
        time.sleep(0.01)

def on_yolo_state_change(change):
    global yolo_live_active
    if change['new'] == 'Live YOLO Inference':
        if not yolo_live_active:
            yolo_live_active = True
            t = threading.Thread(target=yolo_inference_loop, daemon=True)
            t.start()
            status_label.value = '🟢 Real-time YOLO inference running...'
    else:
        yolo_live_active = False
        status_label.value = '🔴 Inference stopped.'

state_widget.observe(on_yolo_state_change, names='value')

# Layout Assembly
center_box = ipywidgets.VBox([
    yolo_preview_widget,
    state_widget
], layout=Layout(align_items='center', margin='0px 0px 15px 0px'))

metrics_box = ipywidgets.HBox([fps_display, latency_display, detections_display])
controls_box = ipywidgets.VBox([
    conf_slider,
    metrics_box,
    status_label
], layout=Layout(align_items='center'))

display(ipywidgets.VBox([center_box, controls_box]))
